# Amazon ML Challenge 2026 - Phase 2 Interactive Debugging

Interactive step-by-step debugging notebook for Phase 2 candidate generation:
- Loads the `train` split Parquet datasets with `id_map`.
- Isolates a single country (`India`) for focused debugging.
- Runs character 3-4 n-gram TF-IDF vectorization on coalesced business names.
- Profiles `sparse_dot_topn.sp_matmul_topn` C++ multithreaded sparse matrix multiplication with `%%time`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import time
import polars as pl
import sparse_dot_topn
from sklearn.feature_extraction.text import TfidfVectorizer

from src.blocking import MultiChannelBlocker
from scripts.03_candidate_generation import load_dataset_split

In [ ]:
parquet_dir = Path("../data/parquet")
id_map = pl.read_parquet(parquet_dir / "id_map.parquet")
s1_df, pool_df = load_dataset_split(parquet_dir, "train", id_map)

countries = s1_df.select("country").unique().drop_nulls().get_column("country").to_list()
print("Countries:", countries)

In [ ]:
country = "India"
s1_c = s1_df.filter(pl.col("country") == country)
pool_c = pool_df.filter(pl.col("country") == country)

print(f"S1 count: {s1_c.height}")
print(f"Pool count: {pool_c.height}")

In [ ]:
# Coalesce text fields
s1_cols = s1_c.columns
s1_exprs = [pl.col(c) for c in ["name_no_legal", "name_clean", "business_name"] if c in s1_cols]

pool_cols = pool_c.columns
pool_exprs = [pl.col(c) for c in ["name_no_legal", "name_clean", "business_name"] if c in pool_cols]

print("Extracting S1 text...")
s1_text_series = (
    s1_c.select(pl.coalesce(s1_exprs).fill_null("").str.strip_chars().alias("text"))
    .get_column("text")
    .to_list()
)

print("Extracting Pool text...")
pool_text_series = (
    pool_c.select(pl.coalesce(pool_exprs).fill_null("").str.strip_chars().alias("text"))
    .get_column("text")
    .to_list()
)

# Vectorizer initialization
ngram_range = (3, 4)
min_df = 2 if len(pool_text_series) > 100 else 1
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=ngram_range,
    min_df=min_df,
    norm="l2",
    sublinear_tf=True,
)

# Fit and Transform
print("Fitting TF-IDF on pool and transforming S1...")
t0 = time.time()
M_pool = vectorizer.fit_transform(pool_text_series)
M_s1 = vectorizer.transform(s1_text_series)
print(f"TF-IDF vectorization completed in {time.time() - t0:.2f}s")

# Print the shape of the sparse matrices!
print(f"M_s1 shape: {M_s1.shape}")
print(f"M_pool shape: {M_pool.shape}")

In [ ]:
%%time
import os

# Transpose candidate pool for sparse dot-product: (N_s1, D) x (D, N_pool) -> (N_s1, N_pool)
M_pool_T = M_pool.T.tocsr()

top_k = 20
min_similarity = 0.20
n_threads = max(1, os.cpu_count() or 1)

print(f"Running sp_matmul_topn with top_k={top_k}, threshold={min_similarity}, n_threads={n_threads}...")
res_csr = sparse_dot_topn.sp_matmul_topn(
    M_s1,
    M_pool_T,
    top_n=top_k,
    threshold=min_similarity,
    sort=True,
    n_threads=n_threads,
)
print("Finished!")
print(f"Result CSR shape: {res_csr.shape}, non-zero elements (nnz): {res_csr.nnz:,}")